# no_strongaug 재학습 (= 설정 D) — 파이프라인/데모용 가중치 확보

**런타임: L4 GPU**. LOO 결과상 강증강 제거(no_strongaug)가 OOD 최고(maxvit 0.954).
그 config만 다시 학습해서 **가중치를 다운로드**합니다.

- config: pretrained + 기본증강(standard) + mixup (강증강 제외)
- 6 model-fold (2 model × 3 fold), 약 25~30분
- 셀 1~6 순서대로. 셀2에서 fireimage_clean.zip 업로드.

In [ ]:
# 1) 클론 + 패키지
%cd /content
!rm -rf fireimage_detection
!git clone https://github.com/yuntaewon812/fireimage_detection.git fireimage_detection -q
!pip install timm grad-cam lime scikit-image scipy -q
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2) 데이터 업로드 — fireimage_clean.zip (약 32MB)
from google.colab import files
up = files.upload()
print('업로드:', list(up.keys()))

In [ ]:
# 3) 데이터 압축 해제
%cd /content/fireimage_detection
!python colab_setup.py

In [ ]:
# 4) no_strongaug 만 재학습 (2 model × 3 fold)
%cd /content/fireimage_detection
import time
t0 = time.time()
!python main_ablation_loo.py --variants no_strongaug --seeds 1004 --epochs 15 --patience 5
print(f'\n학습 시간: {(time.time()-t0)/60:.1f}분')

In [ ]:
# 5) 가중치 다운로드 (★이번엔 꼭 받기★) — no_strongaug fold0/1/2
import shutil, os
SRC = 'model_save/fireimage_loo_no_strongaug_s1004'
print('가중치 파일:')
for f in sorted(os.listdir(SRC)):
    fp = os.path.join(SRC, f)
    if os.path.isdir(fp):
        for g in os.listdir(fp):
            print(f'  {f}/{g}  {os.path.getsize(os.path.join(fp,g))/1e6:.1f}MB')
shutil.make_archive('/content/weights_no_strongaug', 'zip', SRC)
from google.colab import files
files.download('/content/weights_no_strongaug.zip')

In [ ]:
# 6) 성능 확인 (OOD-F1)
import csv
def f1(c): return float(c.split('(')[0])
p = 'results/fireimage_loo_no_strongaug_s1004/metrics.csv'
rows = {}
for r in csv.DictReader(open(p, encoding='utf-8')):
    rows[r['model name']] = f1(r['F1 score'])
for m in ['efficientnetv2', 'maxvit']:
    fs = [rows.get(f'{m}_{i}') for i in range(3)]
    print(f'{m}: fold0/1/2 = {fs}  → OOD(최저)={min(fs):.3f}')